## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# NEXT STEP v1.3 - three follow-up experiments on the SAME 600 evidence bundles
1. **Conditions-only control** (`causal_conditions_only`): the LLM is told which standard risk flags apply, with NO policy text, precedent or effect numbers. Tests whether Causal-RAG's AUROC gain is just "the conditions were flagged".
2. **Graded prompt** (`causal_graded`): like the smooth prompt but no TRIGGERED word; each condition shows value and distance to the cutoff. Tests whether the cliff (boundary-flip response = regime-flip response) disappears.
3. **Boundary flips for the baselines** (`no_rag`, `vanilla_rag`): tells us whether the cliff is specific to the causal prompts.

Needs the artifacts of your v1.2 run attached as a dataset (`evidence.pkl`, `eval_sample.parquet`, `lc_clean.parquet`, `calf_scores.parquet`, ...). Saves every `CHUNK` applicants and resumes.

In [ ]:
import pickle, numpy as np, pandas as pd
need = ['evidence.pkl','eval_sample.parquet','lc_clean.parquet','calf_scores.parquet']
missing = [n for n in need if not paths.have(n)]
assert not missing, f"Missing {missing}. Attach the dataset made from your FINAL_RUN_v1_2 Output (its artifacts/ folder), or run FINAL_RUN_v1_2 first."
from causal_rag import pipeline as pl
from causal_rag.llm import LLM
from causal_rag.config import fit_medians, add_features
bundles = pickle.load(open(paths.art('evidence.pkl'), 'rb'))
N_NEXT = 12 if SMOKE else len(bundles); bundles = bundles[:N_NEXT]
samp = pd.read_parquet(paths.art('eval_sample.parquet')).iloc[:N_NEXT]
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr = (df.role=='train').values
d = add_features(df, fit_medians(df, tr)); idx = d[d.role.isin(['train','val'])].reset_index(drop=True)
llm = LLM()
NEW = ['no_rag', 'vanilla_rag', 'causal_conditions_only', 'causal_graded']
sc_new, cf_new = pl.run_llm_stage(llm, bundles, idx, NEW, NEW, boundary_systems=('causal_conditions_only','causal_graded'),
                                  chunk=CHUNK, tag='llm_parts_v13', boundary_only=('no_rag','vanilla_rag'))
sc_new = sc_new.merge(samp, on='case_id'); sc_new.to_parquet(paths.art('llm_scores_v13.parquet')); cf_new.to_parquet(paths.art('cf_audit_v13.parquet'))
display(cf_new.groupby(['system','kind']).size().unstack())

In [ ]:
import json, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from causal_rag import eval as ev, compare
from causal_rag.config import Featurizer, TREATMENTS
y = sc_new.default.values
old = pd.read_parquet(paths.art('llm_scores.parquet')) if paths.have('llm_scores.parquet') else None
if old is not None:
    sc = old.drop(columns=['default'], errors='ignore').merge(sc_new.drop(columns=['no_rag','vanilla_rag']), on='case_id', how='inner')
else:
    sc = sc_new; print('v1.2 llm_scores.parquet not attached: comparing the 4 new systems only')
y = sc.default.values
va = (df.role=='val').values; yy = d.default.values
flat = Featurizer(None).fit(d, tr); X = np.hstack(flat.transform(d))
te_idx = d.set_index('case_id').index.get_indexer(sc.case_id.values)
gbm = ev.gbm_baseline(X[tr], yy[tr], X[va], yy[va], X[te_idx])
calf = pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id').loc[sc.case_id.values]
LLM_SYS = [c for c in sc.columns if c not in ('case_id','default')]
scores = {k: sc[k].values for k in LLM_SYS}
scores['CALF factual risk (tabular)'] = np.mean([calf[f'pf_{tk}'].values for tk in TREATMENTS], axis=0)
scores['LightGBM (tabular ceiling)'] = gbm
B = 300 if SMOKE else 1000
disc = ev.discrimination_table(y, scores, ref='no_rag', B=B); display(disc)
pairs = [('causal_conditions_only','no_rag'), ('causal_rag','causal_conditions_only'), ('causal_precedent_only','causal_conditions_only'),
         ('causal_calf_only','causal_conditions_only'), ('causal_graded','causal_conditions_only'), ('causal_graded','no_rag'),
         ('causal_graded','causal_rag_smooth'), ('causal_graded','causal_rag')]
pw = ev.pairwise_table(y, scores, pairs, B=B); display(pw)

In [ ]:
thr_by = {s: float(np.quantile(sc_new[s].values, 0.8)) for s in NEW}
cft = ev.counterfactual_table(cf_new, thr_by)
# regime-flip means for no_rag / vanilla come from the v1.2 run (this run only did their boundary flips)
prev = compare.previous_runs().get('v1_2_run')
if prev:
    p = pd.read_csv(prev + '/stage08_counterfactual_table.csv').set_index('system')
    for s in cft.system:
        if s in p.index and pd.isna(cft.loc[cft.system==s, 'mean_abs_dP_regime']).all():
            reg = float(p.loc[s, 'mean_abs_dP_regime']); cft.loc[cft.system==s, 'mean_abs_dP_regime_from_v1_2'] = reg
            cft.loc[cft.system==s, 'cliff_ratio_boundary_over_regime'] = cft.loc[cft.system==s, 'mean_abs_dP_boundary'] / reg
display(cft.T)
print('Cliff ratio = |P(decline) change| for a just-over-the-cutoff edit / for a full regime change. ~1 means it reacts to the label, ~0 means smooth.')
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
disc.set_index('system').auroc.plot.barh(ax=ax[0]); ax[0].axvline(.5, c='k', ls=':'); ax[0].set_title('AUROC')
cft.set_index('system').cliff_ratio_boundary_over_regime.plot.bar(ax=ax[1]); ax[1].set_title('cliff ratio (1 = pure cliff)'); ax[1].axhline(1, c='k', ls=':')
plt.tight_layout(); plt.savefig(paths.art('summary_v13.png'), dpi=130); plt.show()
disc.to_csv(paths.art('results_table.csv'), index=False); pw.to_csv(paths.art('pairwise_table.csv'), index=False)
out = compare.export_run('v1_3_run', disc, cft, dict(note='no audit stage in v1.3 (new controls only)'),
                         dict(N=len(y), CHUNK=CHUNK, code_version='1.3.0', systems_new=NEW),
                         extra_files=[paths.art('pairwise_table.csv'), paths.art('summary_v13.png')])
print('exported to', out, '-> copy into the dataset under results/ to keep this run')

### Optional: package this session's artifacts so they are never lost again
Download `crag_artifacts.zip` from the Output panel and upload it as a Kaggle Dataset (e.g. `crag-artifacts`). Attach that dataset in later notebooks: `restore_artifacts()` and the skip-if-exists guards pick everything up.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/crag_artifacts', 'zip', '/kaggle/working', 'artifacts')
print('wrote /kaggle/working/crag_artifacts.zip')